# Preparar una sola tabla de ASSISTments

Este cuaderno se ejecuta **una vez** para unir los archivos fuente, seleccionar una muestra reproducible y guardar un único CSV comprimido. Aquí no se entrena ningún modelo.

El archivo creado será `EDMCup2023/salida/assistments_rf_dataset.csv.gz`. El cuaderno de exposición lo leerá como su única fuente de datos.

**Unidad de análisis:** una fila representa un problema de la evaluación de fin de unidad de un estudiante. `score` es la etiqueta (0 o 1). Los identificadores se conservan para rastrear las filas, pero se excluirán de las variables de Random Forest.

## 1. Preparar Google Drive

Coloca los CSV oficiales dentro de `Mi unidad/EDMCup2023/data/`. El lector busca archivos en subcarpetas, por si `action_logs.csv` está en `data/files/`.

Este cuaderno no edita los archivos fuente. Si el CSV consolidado ya existe, se reutiliza y no se sobrescribe. Si cambias la muestra (`MAX_STUDENTS` o `SEED`) y quieres regenerar el resultado, cambia también el nombre `CANONICAL_CSV` en la celda de configuración, por ejemplo a `assistments_rf_dataset_v2.csv.gz`.


In [ ]:
from pathlib import Path
import gc
import re
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')

SEED = 42
MAX_STUDENTS = 3000   # Muestra reproducible. Usa None para incluir a todos.
CHUNK_SIZE = 300_000  # Lee el archivo grande de acciones por bloques.

BASE_DIR = Path('/content/drive/MyDrive/EDMCup2023')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'salida'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_CSV = OUTPUT_DIR / 'assistments_rf_dataset.csv.gz'

print('Datos crudos:', DATA_DIR)
print('CSV único de referencia:', CANONICAL_CSV)
print(f'Estudiantes en la muestra: {MAX_STUDENTS:,}' if MAX_STUDENTS else 'Se usarán todos los estudiantes')


## 2. Funciones de lectura

Se normalizan los identificadores para que coincidan entre archivos. Los registros de acciones, que suelen ser los más voluminosos, se leen en bloques.

In [ ]:
def find_file(filename, required=True):
    matches = list(DATA_DIR.rglob(filename)) if DATA_DIR.exists() else []
    if not matches:
        if required:
            raise FileNotFoundError(
                f'No encuentro {filename} dentro de {DATA_DIR}. '
                'Descomprime los datos oficiales en esa carpeta y vuelve a ejecutar.'
            )
        return None
    if len(matches) > 1:
        # Prioriza un archivo directamente en data/ y luego el primero por ruta.
        matches.sort(key=lambda p: (len(p.parts), str(p).lower()))
    return matches[0]


def normalize_id(values):
    """Unifica IDs leídos como entero, decimal o texto sin alterar su contenido."""
    return (values.astype('string').str.strip()
            .str.replace(r'\.0$', '', regex=True))


def read_scores(path, split_name):
    df = pd.read_csv(path, low_memory=False)
    required = {'assignment_log_id', 'problem_id', 'score'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name}: faltan columnas {sorted(missing)}')
    df['assignment_log_id'] = normalize_id(df['assignment_log_id'])
    df['problem_id'] = normalize_id(df['problem_id'])
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    df['source_split'] = split_name
    return df


def read_filtered_csv(path, columns, key_column, accepted_ids, chunksize=CHUNK_SIZE):
    """Lee por bloques un CSV y conserva solo IDs solicitados."""
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = set(columns) - set(header)
    if missing:
        raise ValueError(f'{path.name}: faltan columnas {sorted(missing)}')
    parts = []
    dtype = {key_column: 'string'}
    for chunk in pd.read_csv(path, usecols=columns, dtype=dtype,
                             chunksize=chunksize, low_memory=False):
        chunk[key_column] = normalize_id(chunk[key_column])
        chunk = chunk[chunk[key_column].isin(accepted_ids)]
        if not chunk.empty:
            parts.append(chunk)
    if not parts:
        return pd.DataFrame(columns=columns)
    return pd.concat(parts, ignore_index=True).drop_duplicates()


def read_assignment_details(path, assignment_ids):
    columns = ['assignment_log_id', 'student_id', 'sequence_id', 'assignment_end_time']
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = set(columns) - set(header)
    if missing:
        raise ValueError(f'{path.name}: faltan columnas {sorted(missing)}')
    details = read_filtered_csv(path, columns, 'assignment_log_id', assignment_ids)
    for col in ['student_id', 'sequence_id']:
        details[col] = normalize_id(details[col])
    details['assignment_end_time'] = details['assignment_end_time'].replace(r'^\s*$', pd.NA, regex=True)
    return details.drop_duplicates('assignment_log_id', keep='first')


def normalize_category(values):
    return (values.astype('string').fillna('other').str.strip().str.lower()
            .str.replace(r'[\s-]+', '_', regex=True)
            .str.replace(r'[^a-z0-9_]+', '', regex=True)
            .replace('', 'other'))


def clean_text(values):
    return (values.astype('string').fillna('NA').str.lower()
            .str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
            .str.replace(r'\s+', ' ', regex=True).str.strip())

## 3. Construir y guardar el CSV único

La muestra se elige por estudiante con semilla 42. Aumenta `MAX_STUDENTS` para conservar más estudiantes; usa `None` para procesar todos. El CSV se comprime con gzip para reducir el espacio ocupado y pandas lo abre directamente.

In [ ]:
if CANONICAL_CSV.exists():
    print(f'Ya existe {CANONICAL_CSV}. Se reutilizará y no se sobrescribirá.')
    print('Para construir otra muestra, cambia el nombre de CANONICAL_CSV en la celda de configuración.')
else:
    print('Leyendo puntuaciones y seleccionando estudiantes...')
    # Archivos oficiales de entrada
    training_path = find_file('training_unit_test_scores.csv')
    evaluation_path = find_file('evaluation_unit_test_scores.csv')
    total_scores_path = find_file('unit_test_scores.csv', required=False)
    relationships_path = find_file('assignment_relationships.csv')
    assignment_path = find_file('assignment_details.csv')
    sequence_path = find_file('sequence_details.csv')
    problem_path = find_file('problem_details.csv')
    action_path = find_file('action_logs.csv')  # Puede estar en data/files/

    train_scores = read_scores(training_path, 'train')
    eval_scores = read_scores(evaluation_path, 'evaluation')
    all_target_ids = set(pd.concat([
        train_scores['assignment_log_id'], eval_scores['assignment_log_id']
    ]).dropna().unique())

    # Primer barrido de assignment_details: solo necesitamos identificar estudiantes
    target_details = read_assignment_details(assignment_path, all_target_ids)
    student_by_assignment = target_details[['assignment_log_id', 'student_id']].dropna()
    known_student_ids = student_by_assignment['student_id'].drop_duplicates()
    if known_student_ids.empty:
        raise ValueError('No se encontraron student_id para las evaluaciones finales.')

    if MAX_STUDENTS is None or MAX_STUDENTS >= len(known_student_ids):
        selected_students = set(known_student_ids)
    else:
        selected_students = set(known_student_ids.sample(
            n=MAX_STUDENTS, random_state=SEED
        ))

    selected_assignment_ids = set(
        student_by_assignment.loc[
            student_by_assignment['student_id'].isin(selected_students),
            'assignment_log_id'
        ]
    )
    targets = pd.concat([train_scores, eval_scores], ignore_index=True, sort=False)
    targets = targets[targets['assignment_log_id'].isin(selected_assignment_ids)].copy()
    targets = targets.merge(student_by_assignment, on='assignment_log_id', how='left')
    targets = targets.drop_duplicates(['assignment_log_id', 'problem_id', 'source_split'])
    targets = targets.rename(columns={'assignment_log_id': 'unit_test_assignment_log_id'})

    # Si la descarga incluye etiquetas finales del conjunto de evaluación, se usan para
    # medir el resultado oficial. Si no, la evaluación se hará con una partición interna.
    if total_scores_path is not None and targets.loc[
        targets['source_split'].eq('evaluation'), 'score'
    ].isna().any():
        eval_rows = targets[targets['source_split'].eq('evaluation')]
        eval_pairs = set(zip(eval_rows['unit_test_assignment_log_id'], eval_rows['problem_id']))
        score_parts = []
        for chunk in pd.read_csv(
            total_scores_path,
            usecols=['assignment_log_id', 'problem_id', 'score'],
            dtype={'assignment_log_id': 'string', 'problem_id': 'string'},
            chunksize=CHUNK_SIZE,
            low_memory=False
        ):
            chunk['assignment_log_id'] = normalize_id(chunk['assignment_log_id'])
            chunk['problem_id'] = normalize_id(chunk['problem_id'])
            pairs = list(zip(chunk['assignment_log_id'], chunk['problem_id']))
            chunk = chunk.loc[[pair in eval_pairs for pair in pairs]]
            if not chunk.empty:
                score_parts.append(chunk)
        if score_parts:
            score_lookup = pd.concat(score_parts, ignore_index=True)
            score_lookup['score'] = pd.to_numeric(score_lookup['score'], errors='coerce')
            score_lookup = score_lookup.drop_duplicates(['assignment_log_id', 'problem_id'])
            lookup = dict(zip(
                zip(score_lookup['assignment_log_id'], score_lookup['problem_id']),
                score_lookup['score']
            ))
            mask = targets['source_split'].eq('evaluation') & targets['score'].isna()
            targets.loc[mask, 'score'] = [
                lookup.get((a, p), np.nan) for a, p in zip(
                    targets.loc[mask, 'unit_test_assignment_log_id'],
                    targets.loc[mask, 'problem_id']
                )
            ]

    target_ids = set(targets['unit_test_assignment_log_id'].dropna())
    rel_columns = ['unit_test_assignment_log_id', 'in_unit_assignment_log_id']
    relationships = read_filtered_csv(
        relationships_path, rel_columns, 'unit_test_assignment_log_id', target_ids
    )
    relationships['in_unit_assignment_log_id'] = normalize_id(
        relationships['in_unit_assignment_log_id']
    )
    target_columns = [
        'unit_test_assignment_log_id', 'problem_id', 'score', 'student_id', 'source_split'
    ]
    target_rel = targets[target_columns].merge(
        relationships, on='unit_test_assignment_log_id', how='left'
    )
    in_unit_ids = set(target_rel['in_unit_assignment_log_id'].dropna())

    print('Filtrando prácticas y variables del curso...')
    # Segundo barrido: detalles solo de evaluaciones y prácticas seleccionadas
    needed_assignment_ids = target_ids | in_unit_ids
    assignment_details = read_assignment_details(assignment_path, needed_assignment_ids)

    # Relación de prácticas no terminadas por evaluación final
    rel_detail = relationships.merge(
        assignment_details[['assignment_log_id', 'assignment_end_time']],
        left_on='in_unit_assignment_log_id', right_on='assignment_log_id', how='left'
    )
    rel_detail = rel_detail.drop_duplicates(
        ['unit_test_assignment_log_id', 'in_unit_assignment_log_id']
    )
    rel_detail['unfinished'] = rel_detail['assignment_end_time'].isna().astype('uint8')
    assignment_stats = rel_detail.groupby('unit_test_assignment_log_id').agg(
        total_practice=('in_unit_assignment_log_id', 'nunique'),
        unfinished_practice=('unfinished', 'sum')
    ).reset_index()
    assignment_stats['notfinish_percent'] = (
        assignment_stats['unfinished_practice'] /
        assignment_stats['total_practice'].replace(0, np.nan)
    ).round(4)

    # Extraer acciones por bloques y resumirlas por evaluación final + problema
    action_categories = [
        'answer_requested', 'assignment_finished', 'assignment_resumed',
        'assignment_started', 'continue_selected', 'correct_response',
        'explanation_requested', 'hint_requested', 'live_tutor_requested',
        'open_response', 'problem_finished', 'problem_started',
        'skill_related_video_requested', 'wrong_response'
    ]
    tutoring_categories = ['answer', 'explanation', 'hint', 'no_tutoring']
    action_columns = [
        'assignment_log_id', 'problem_id', 'max_attempts', 'available_core_tutoring',
        'score_viewable', 'continuous_score_viewable', 'action'
    ]
    action_header = pd.read_csv(action_path, nrows=0).columns.tolist()
    missing_action = set(action_columns) - set(action_header)
    if missing_action:
        raise ValueError(f'action_logs.csv: faltan columnas {sorted(missing_action)}')

    action_map = target_rel[
        ['unit_test_assignment_log_id', 'problem_id', 'in_unit_assignment_log_id']
    ].dropna(subset=['in_unit_assignment_log_id']).drop_duplicates()
    action_parts = []
    mode_parts = {col: [] for col in [
        'max_attempts', 'score_viewable', 'continuous_score_viewable'
    ]}

    print('Procesando action_logs.csv por bloques...')
    for chunk_number, action_chunk in enumerate(pd.read_csv(
        action_path, usecols=action_columns,
        dtype={'assignment_log_id': 'string'}, chunksize=CHUNK_SIZE,
        low_memory=False
    ), start=1):
        if chunk_number == 1 or chunk_number % 10 == 0:
            print(f'Bloque de acciones procesado: {chunk_number}')
        action_chunk['assignment_log_id'] = normalize_id(action_chunk['assignment_log_id'])
        action_chunk['problem_id'] = normalize_id(action_chunk['problem_id'])
        action_chunk = action_chunk[action_chunk['assignment_log_id'].isin(in_unit_ids)]
        if action_chunk.empty:
            continue
        # Estos campos describen el problema de práctica y suelen ser constantes por problem_id.
        constant_fields = [
            'max_attempts', 'available_core_tutoring', 'score_viewable',
            'continuous_score_viewable'
        ]
        for col in constant_fields:
            action_chunk[col] = action_chunk.groupby('problem_id')[col].transform('first')
        joined = action_map.merge(
            action_chunk.rename(columns={'problem_id': 'in_unit_problem_id'}),
            left_on='in_unit_assignment_log_id', right_on='assignment_log_id', how='inner'
        )
        if joined.empty:
            continue
        group_keys = ['unit_test_assignment_log_id', 'problem_id']

        for raw_col, prefix, allowed in [
            ('action', 'action', action_categories),
            ('available_core_tutoring', 'available_core_tutoring', tutoring_categories)
        ]:
            value = normalize_category(joined[raw_col])
            value = value.where(value.isin(allowed), 'other')
            long = joined[group_keys].copy()
            long['feature'] = prefix + '_' + value
            wide = long.groupby(group_keys + ['feature']).size().unstack(fill_value=0)
            action_parts.append(wide.astype('uint32'))

        for col in mode_parts:
            value = pd.to_numeric(joined[col], errors='coerce').fillna(-1)
            mode_frame = joined[group_keys].copy()
            mode_frame[col] = value
            counts = mode_frame.groupby(group_keys + [col]).size().rename('n').reset_index()
            mode_parts[col].append(counts)

    if action_parts:
        action_features = pd.concat(action_parts).groupby(level=[0, 1]).sum().reset_index()
    else:
        action_features = pd.DataFrame(columns=['unit_test_assignment_log_id', 'problem_id'])

    mode_features = None
    for col, parts in mode_parts.items():
        if not parts:
            continue
        counts = pd.concat(parts, ignore_index=True).groupby(
            ['unit_test_assignment_log_id', 'problem_id', col], as_index=False
        )['n'].sum()
        counts = counts.sort_values(['n', col]).drop_duplicates(
            ['unit_test_assignment_log_id', 'problem_id'], keep='last'
        )[['unit_test_assignment_log_id', 'problem_id', col]]
        mode_features = counts if mode_features is None else mode_features.merge(
            counts, on=['unit_test_assignment_log_id', 'problem_id'], how='outer'
        )
    if mode_features is None:
        mode_features = pd.DataFrame(columns=[
            'unit_test_assignment_log_id', 'problem_id', 'max_attempts',
            'score_viewable', 'continuous_score_viewable'
        ])

    # Tabla base: una fila por resultado final y problema
    dataset = targets[target_columns].drop_duplicates(
        ['unit_test_assignment_log_id', 'problem_id', 'source_split']
    ).copy()
    dataset = dataset.merge(action_features, on=['unit_test_assignment_log_id', 'problem_id'], how='left')
    dataset = dataset.merge(mode_features, on=['unit_test_assignment_log_id', 'problem_id'], how='left')
    dataset = dataset.merge(assignment_stats[['unit_test_assignment_log_id', 'notfinish_percent']],
                            on='unit_test_assignment_log_id', how='left')

    # Variables de secuencia: niveles de ruta curricular 1 a 4
    sequence_fields = [
        'sequence_folder_path_level_1', 'sequence_folder_path_level_2',
        'sequence_folder_path_level_3', 'sequence_folder_path_level_4'
    ]
    sequence_header = pd.read_csv(sequence_path, nrows=0).columns.tolist()
    missing_sequence = set(['sequence_id'] + sequence_fields) - set(sequence_header)
    if missing_sequence:
        raise ValueError(f'sequence_details.csv: faltan columnas {sorted(missing_sequence)}')
    sequence_ids = set(assignment_details['sequence_id'].dropna())
    sequence_details = read_filtered_csv(
        sequence_path, ['sequence_id'] + sequence_fields, 'sequence_id', sequence_ids
    )
    sequence_details['sequence_id'] = normalize_id(sequence_details['sequence_id'])
    target_sequence = dataset[['unit_test_assignment_log_id']].drop_duplicates().merge(
        assignment_details[['assignment_log_id', 'sequence_id']],
        left_on='unit_test_assignment_log_id', right_on='assignment_log_id', how='left'
    )
    dataset = dataset.merge(
        target_sequence[['unit_test_assignment_log_id', 'sequence_id']],
        on='unit_test_assignment_log_id', how='left'
    ).merge(sequence_details, on='sequence_id', how='left')
    for col in sequence_fields:
        dataset[col] = clean_text(dataset[col]).replace('', 'na').fillna('na')
    sequence_dummies = pd.get_dummies(
        dataset[sequence_fields], prefix=sequence_fields, dtype=np.uint8
    )
    dataset = pd.concat([dataset.drop(columns=sequence_fields + ['sequence_id']), sequence_dummies], axis=1)

    # Variables del problema final: categorías, código de habilidad y vector de texto ya provisto
    problem_fields = [
        'problem_id', 'problem_type', 'problem_multipart_position', 'problem_skill_code',
        'problem_skill_description', 'problem_multipart_id', 'problem_text_bert_pca'
    ]
    problem_header = pd.read_csv(problem_path, nrows=0).columns.tolist()
    missing_problem = set(problem_fields) - set(problem_header)
    if missing_problem:
        raise ValueError(f'problem_details.csv: faltan columnas {sorted(missing_problem)}')
    problem_ids = set(dataset['problem_id'].dropna())
    problem_details = read_filtered_csv(problem_path, problem_fields, 'problem_id', problem_ids)
    problem_details['problem_id'] = normalize_id(problem_details['problem_id'])
    problem_details = problem_details.drop_duplicates('problem_id', keep='first')
    dataset = dataset.merge(problem_details, on='problem_id', how='left')

    skill_parts = dataset['problem_skill_code'].fillna('NA').astype('string').str.split('.', n=3, expand=True)
    skill_parts = skill_parts.reindex(columns=range(4))
    skill_parts.columns = ['skill_code1', 'skill_code2', 'skill_code3', 'skill_code4']
    dataset = pd.concat([dataset, skill_parts], axis=1)

    problem_categories = [
        'problem_type', 'problem_multipart_position', 'skill_code1', 'skill_code2',
        'skill_code3', 'skill_code4', 'problem_skill_description'
    ]
    for col in problem_categories:
        dataset[col] = clean_text(dataset[col]).replace('', 'na').fillna('na')
    problem_dummies = pd.get_dummies(
        dataset[problem_categories], prefix=problem_categories, dtype=np.uint8
    )

    def parse_bert_vector(value, width=32):
        result = np.full(width, np.nan, dtype=np.float32)
        if pd.isna(value):
            return result
        numbers = re.findall(r'[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?', str(value))
        try:
            parsed = np.asarray([float(x) for x in numbers[:width]], dtype=np.float32)
        except ValueError:
            return result
        result[:len(parsed)] = parsed
        return result

    bert_matrix = np.vstack(dataset['problem_text_bert_pca'].map(parse_bert_vector).to_numpy())
    bert_columns = [f'problem_text_bert_pca_{i}' for i in range(bert_matrix.shape[1])]
    bert_features = pd.DataFrame(bert_matrix, columns=bert_columns, index=dataset.index)
    multipart_frequency = dataset['problem_multipart_id'].value_counts(dropna=False)
    dataset['multipart_id_frequency'] = dataset['problem_multipart_id'].map(multipart_frequency).fillna(0)

    raw_problem_columns = list(dict.fromkeys(
        problem_fields[1:] + problem_categories[1:] + ['problem_multipart_id']
    ))
    dataset = pd.concat([
        dataset.drop(columns=raw_problem_columns, errors='ignore'),
        problem_dummies,
        bert_features
    ], axis=1)

    # Tratamiento final: contadores de acción en 0; faltantes de otras variables en -1.
    action_feature_columns = [c for c in dataset.columns if c.startswith('action_') or
                              c.startswith('available_core_tutoring_')]
    dataset[action_feature_columns] = dataset[action_feature_columns].fillna(0)
    dataset = dataset.replace([np.inf, -np.inf], np.nan)
    feature_columns = [c for c in dataset.columns if c not in {
        'unit_test_assignment_log_id', 'problem_id', 'student_id', 'source_split', 'score'
    }]
    dataset[feature_columns] = dataset[feature_columns].apply(pd.to_numeric, errors='coerce')
    dataset[feature_columns] = dataset[feature_columns].fillna(-1).astype('float32')
    dataset['score'] = pd.to_numeric(dataset['score'], errors='coerce').astype('float32')

    # Guardado nuevo: gzip ahorra espacio y deja un único CSV de referencia.
    if dataset.empty:
        raise ValueError('La tabla final quedó vacía; revisa que los CSV oficiales correspondan al EDM Cup 2023.')
    dataset.to_csv(CANONICAL_CSV, index=False, compression='gzip')
    size_mb = CANONICAL_CSV.stat().st_size / (1024 ** 2)
    print(f'CSV único creado: {CANONICAL_CSV}')
    print(f'Filas: {len(dataset):,} | estudiantes: {dataset.student_id.nunique():,} | columnas: {dataset.shape[1]:,}')
    print(f'Tamaño comprimido: {size_mb:.1f} MB')
    del train_scores, eval_scores, targets, target_rel, relationships, action_parts, mode_parts
    gc.collect()

## 4. Comprobar el resultado

Se muestran unas filas y variables de referencia del CSV terminado. Este resumen permite explicar qué cambió entre los archivos fuente y la tabla lista para modelar.


In [ ]:
if not CANONICAL_CSV.exists():
    raise FileNotFoundError(f'No se generó el CSV esperado: {CANONICAL_CSV}')
preview = pd.read_csv(CANONICAL_CSV, nrows=5, low_memory=False)
print(f'Columnas: {len(pd.read_csv(CANONICAL_CSV, nrows=0).columns):,}')
print('Variables de rastreo, partición y etiqueta:')
display(preview[['student_id', 'unit_test_assignment_log_id', 'problem_id', 'source_split', 'score']])
print('Primeras variables predictoras:')
feature_preview = [c for c in preview.columns if c not in {
    'student_id', 'unit_test_assignment_log_id', 'problem_id', 'source_split', 'score'
}][:8]
display(preview[feature_preview])


## Variables que quedan en el CSV

- **Etiqueta:** `score` (0 = resultado no exitoso, 1 = exitoso).
- **Rastreo:** identificador del estudiante, evaluación, problema y partición (`train` o `evaluation`). No se usarán como predictores.
- **Actividad:** conteos de acciones observadas en las prácticas relacionadas y porcentaje de prácticas sin finalizar.
- **Curso y problema:** niveles de la secuencia curricular, características del problema y habilidad.
- **Texto del problema:** se conservan las 32 dimensiones `problem_text_bert_pca` ya disponibles en la fuente.

Las variables categóricas se codifican y los faltantes de predictores se representan numéricamente. La muestra permite ejecutar la exposición en menos tiempo; puede cambiar las métricas frente al artículo. El artículo emplea más pasos y compara varios clasificadores, mientras que el cuaderno de exposición entrenará solo Random Forest.